In [ ]:
import random
import json
import re
import asyncio
import pandas as pd
import os
from tqdm.asyncio import tqdm_asyncio
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_score
from vpei.common_variables import *
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS
from vpei.epistemic_consistency.experiment_types import carry_out_absolute_experiment
from vpei.epistemic_consistency.experiment_utils import print_absolute_experiment_results

def randomly_flip_digits(answer):
    def flip_digit(match):
        return str(random.randint(0, 9))
    return re.sub(r'\d', flip_digit, answer)

df = pd.read_csv("./data/sample_physics_problems_and_solutions.csv")
df['physics_solution'] = df['answers'] + " " + df['unit'].fillna('')
df.rename(columns={"problem": "physics_problem"}, inplace=True)
df['perturbed_physics_solution'] = df['physics_solution'].apply(randomly_flip_digits)
# For absolute experiment, we use correct solutions
df

In [ ]:
experiment_name = "physics_problems"
system_prompt = EXPERIMENTS[experiment_name]["absolute_experiment"]["system_prompt"]
user_prompt_template = EXPERIMENTS[experiment_name]["absolute_experiment"]["user_prompt_template"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template)

In [ ]:
# model_name = "gpt-4o-mini"
model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})
name = "J.S."
physics_problem = df.iloc[0]['physics_problem']
physics_solution = df.iloc[0]['physics_solution']
user_prompt = user_prompt_template.format(name=name, political_attitude="Republican", physics_problem=physics_problem, physics_solution=physics_solution)
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]
response = make_llm_request(model_name, messages, **model_kwargs)
print("Response:", response)

In [ ]:
models = ["gpt-5-mini"]

n = 3
stimuli_factors = ["physics_problem", "physics_solution"]
additional_variables_from_df_to_save = []
custom_model_kwargs = {}
random_seed = 43
path_to_save_model_outputs = "./absolute_experiment/"

In [ ]:
payloads = await carry_out_absolute_experiment(models=models, df=df, n=n, system_prompt=system_prompt, user_prompt_template=user_prompt_template, stimuli_factors=stimuli_factors, 
                                               additional_variables_from_df_to_save=additional_variables_from_df_to_save, custom_model_kwargs=custom_model_kwargs, 
                                               path_to_save_model_outputs=path_to_save_model_outputs, random_seed=random_seed)

print_absolute_experiment_results(payloads, models)